[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/openai/openai_agents_prompt_chaining.ipynb)


# <center>OpenAI 智能体模式：提示词链</center>

本入门指南介绍如何使用 `openai-agents` 库将两个提示词串联起来生成输出。

下面使用这一模式构建股票投资组合生成系统：
1. **搜索智能体（生成）**：在网上搜索指定股票代码的信息。
2. **报告智能体（生成）**：创建符合用户投资策略的股票和 ETF 投资组合。

### 安装库

In [ ]:
# 安装 OpenAI 基础库
!pip install -q openai openai-agents pydantic

# 安装 OpenInference/OpenTelemetry 追踪的可选库
!pip install -q arize-phoenix-otel openinference-instrumentation-openai-agents openinference-instrumentation-openai

### 设置密钥
将 OpenAI API 密钥添加到环境变量 `OPENAI_API_KEY`。

从 [app.phoenix.arize.com](https://app.phoenix.arize.com) 的设置页面复制 Phoenix `API_KEY`。

In [ ]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("🔑 请输入 OpenAI API 密钥： ")

if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

### 设置追踪

In [ ]:
from phoenix.otel import register

tracer_provider = register(
    project_name="openai-agents",
    auto_instrument=True,
)

## 创建智能体

In [ ]:
from pprint import pprint
from textwrap import dedent

from agents import Agent, Runner, TResponseInputItem, WebSearchTool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field


class PortfolioItem(BaseModel):
    ticker: str = Field(description="股票或 ETF 的代码。")
    allocation: float = Field(
        description="该代码在投资组合中的分配百分比。所有分配比例总和应为 100。"
    )
    reason: str = Field(description="将该代码纳入投资组合的原因。")


class Portfolio(BaseModel):
    tickers: list[PortfolioItem] = Field(
        description="能够支持用户声明的投资策略的股票代码列表。"
    )


portfolio_agent = Agent(
    name="Portfolio Agent",
    instructions=dedent(
        """你是资深金融分析师。你会收到股票研究报告。请创建能够支持用户声明的投资策略的股票和 ETF 投资组合，并在分配理由中引用研究报告的事实和数据。"""
    ),
    model="o4-mini",
    output_type=Portfolio,
)

research_agent = Agent(
    name="FinancialSearchAgent",
    instructions=dedent(
        """你是专注金融主题的研究助手。给定投资策略后，使用网络搜索获取最新背景，生成不超过 50 个词的简短摘要，介绍支持该策略的股票，重点关注对金融分析师有用的关键数字、事件或引述。"""
    ),
    model="gpt-4.1",
    tools=[WebSearchTool()],
    model_settings=ModelSettings(tool_choice="required", parallel_tool_calls=True),
)

user_input = input("请输入投资策略： ")
input_items: list[TResponseInputItem] = [
    {"content": f"我的投资策略： {user_input}", "role": "user"}
]


research_output = await Runner.run(research_agent, input_items)
pprint(research_output.final_output)

input_items = research_output.to_input_list()
portfolio_output = await Runner.run(portfolio_agent, input_items)
pprint(portfolio_output.final_output)